In [1]:
# Se define la carga de los datos que alimentan el taller.

def load_data():

    import pandas as pd

    SENTENCES = "../data/sentences.csv.gz"

    dataframe = pd.read_csv(
        SENTENCES,
        index_col=False,
        compression="zip",
    )

    data = dataframe.phrase
    target = dataframe.target

    return data, target

In [2]:
# Se define la separación entre entrenamiento y prueba.

def make_train_test_split(x, y):

    from sklearn.model_selection import train_test_split

    (x_train, x_test, y_train, y_test) = train_test_split(
        x,
        y,
        test_size=0.25,
        random_state=123456,
    )
    return x_train, x_test, y_train, y_test

In [3]:
# Se define el pipeline para conservar las transformaciones junto al modelo.

def make_pipeline(estimator):

    from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
    from sklearn.pipeline import Pipeline

    vectorizer = CountVectorizer(
        lowercase=True,
        analyzer="word",
        token_pattern=r"\b[a-zA-Z]\w+\b",
        stop_words="english",
    )

    transformer = TfidfTransformer()

    pipeline = Pipeline(
        steps=[
            ("vectorizer", vectorizer),
            ("transformer", transformer),
            ("estimator", estimator),
        ],
        verbose=False,
    )

    return pipeline

In [4]:
# Se define la búsqueda de parámetros mediante validación cruzada.

def make_grid_search(estimator, param_grid, cv=5):

    from sklearn.model_selection import GridSearchCV

    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring="balanced_accuracy",
    )

    return grid_search

In [5]:
# Se define la carga del estimador almacenado.

def load_estimator():

    import os
    import pickle

    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_PATH = f"{OUTPUT_FOLDER}/estimator.pkl"

    if not os.path.exists(ESTIMATOR_PATH):
        return None

    with open(ESTIMATOR_PATH, "rb") as file:
        estimator = pickle.load(file)

    return estimator

In [6]:
# Se define el almacenamiento del estimador para reutilizarlo posteriormente.

def save_estimator(estimator, x_test, y_test):

    import pickle

    from sklearn.metrics import balanced_accuracy_score

    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_PATH = f"{OUTPUT_FOLDER}/estimator.pkl"

    best_estimator = load_estimator()

    if best_estimator is not None:

        saved_balanced_accuracy = balanced_accuracy_score(
            y_true=y_test, y_pred=best_estimator.predict(x_test)
        )

        current_balanced_accuracy = balanced_accuracy_score(
            y_true=y_test, y_pred=estimator.predict(x_test)
        )

        if current_balanced_accuracy < saved_balanced_accuracy:
            estimator = best_estimator

    with open(ESTIMATOR_PATH, "wb") as file:
        pickle.dump(estimator, file)

In [7]:
# Se define el entrenamiento de la regresión logística.

def train_logistic_regression():

    from sklearn.linear_model import LogisticRegression

    data, target = load_data()

    x_train, x_test, y_train, y_test = make_train_test_split(
        x=data,
        y=target,
    )

    pipeline = make_pipeline(
        estimator=LogisticRegression(max_iter=1000),
    )

    estimator = make_grid_search(
        estimator=pipeline,
        param_grid={
            "transformer__norm": ["l1", "l2", None],
            "transformer__use_idf": [True, False],
            "transformer__smooth_idf": [True, False],
        },
        cv=5,
    )

    estimator.fit(x_train, y_train)

    save_estimator(estimator, x_test, y_test)


train_logistic_regression()

FileNotFoundError: [Errno 2] No such file or directory: '../data/sentences.csv.gz'

In [ ]:
# Se define el uso del estimador ya almacenado.

def use_estimator():

    import pickle

    import pandas as pd

    SENTENCES = "../data/sentences.csv.gz"
    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_PATH = f"{OUTPUT_FOLDER}/estimator.pkl"

    dataframe = pd.read_csv(
        SENTENCES,
        index_col=False,
        compression="zip",
    )

    data = dataframe.phrase

    with open(ESTIMATOR_PATH, "rb") as file:
        estimator = pickle.load(file)

    prediction = estimator.predict(data)

    return prediction


use_estimator()

In [ ]:
# Se define la verificación del estimador que quedó almacenado.

def check_estimator():

    import pickle

    import pandas as pd
    from sklearn.metrics import accuracy_score, balanced_accuracy_score

    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_PATH = f"{OUTPUT_FOLDER}/estimator.pkl"

    data, target = load_data()

    x_train, x_test, y_train_true, y_test_true = make_train_test_split(
        x=data,
        y=target,
    )

    with open(ESTIMATOR_PATH, "rb") as file:
        estimator = pickle.load(file)

    y_train_pred = estimator.predict(x_train)
    y_test_pred = estimator.predict(x_test)

    accuracy_train = round(accuracy_score(y_train_true, y_train_pred), 4)
    accuracy_test = round(accuracy_score(y_test_true, y_test_pred), 4)
    balanced_accuracy_train = round(
        balanced_accuracy_score(y_train_true, y_train_pred), 4
    )
    balanced_accuracy_test = round(balanced_accuracy_score(y_test_true, y_test_pred), 4)

    print(estimator.best_estimator_, ":", sep="")
    print(f"  Balanced Accuracy: {balanced_accuracy_test} ({balanced_accuracy_train})")
    print(f"           Accuracy: {accuracy_test} ({accuracy_train})")


check_estimator()

In [ ]:
# Se define el entrenamiento del clasificador neuronal para compararlo.

def train_mlp_classifier():

    from sklearn.metrics import balanced_accuracy_score
    from sklearn.neural_network import MLPClassifier

    data, target = load_data()

    x_train, x_test, y_train, y_test = make_train_test_split(
        x=data,
        y=target,
    )

    pipeline = make_pipeline(
        estimator=MLPClassifier(max_iter=10000),
    )

    estimator = make_grid_search(
        estimator=pipeline,
        param_grid={
            "transformer__norm": ["l1", "l2", None],
            "transformer__use_idf": [True, False],
            "transformer__smooth_idf": [True, False],
            "estimator__hidden_layer_sizes": [(1,), (5,)],
            "estimator__solver": ["adam"],
            "estimator__learning_rate_init": [0.01, 0.001],
        },
        cv=5,
    )

    estimator.fit(x_train, y_train)

    save_estimator(estimator, x_test, y_test)


# train_mlp_classifier()

In [ ]:
# Se ejecuta la verificación del estimador almacenado.

check_estimator()